In [1]:
import numpy as np
import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)

In [20]:
df = pd.read_csv("../data/raw/sales_data_raw.csv")

In [4]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [5]:
df.shape

(76000, 16)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76000 entries, 0 to 75999
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76000 non-null  object 
 1   Store ID            76000 non-null  object 
 2   Product ID          76000 non-null  object 
 3   Category            76000 non-null  object 
 4   Region              76000 non-null  object 
 5   Inventory Level     76000 non-null  int64  
 6   Units Sold          76000 non-null  int64  
 7   Units Ordered       76000 non-null  int64  
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  int64  
 10  Weather Condition   76000 non-null  object 
 11  Promotion           76000 non-null  int64  
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  object 
 14  Epidemic            76000 non-null  int64  
 15  Demand              76000 non-null  int64  
dtypes: f

In [7]:
df.describe()

,Inventory Level,Units Sold,Units Ordered,Price,Discount,Promotion,Competitor Pricing,Epidemic,Demand
count,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000,76000.000000
mean,301.062842,88.827316,89.090645,67.726028,9.087039,0.328947,69.454029,0.200000,104.317158
std,226.510161,43.994525,162.404627,39.377899,7.475781,0.469834,40.943818,0.400003,46.964801
min,0.000000,0.000000,0.000000,4.740000,0.000000,0.000000,4.290000,0.000000,4.000000
25%,136.000000,58.000000,0.000000,31.997500,5.000000,0.000000,32.620000,0.000000,71.000000
50%,227.000000,84.000000,0.000000,64.500000,10.000000,0.000000,65.700000,0.000000,100.000000
75%,408.000000,114.000000,121.000000,95.830000,10.000000,1.000000,97.932500,0.000000,133.000000
max,2267.000000,426.000000,1616.000000,228.030000,25.000000,1.000000,261.220000,1.000000,430.000000


In [8]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

In [9]:
categorical_cols = [
    'Category', 'Region', 'Weather Condition',
    'Promotion', 'Seasonality', 'Epidemic'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

In [10]:
# Numeric columns
num_cols = df.select_dtypes(include=['float64','int64']).columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)


In [11]:
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]

print(f"Removed {before - after} duplicate rows")


Removed 0 duplicate rows


In [12]:
def cap_outliers(col):
    Q1 = col.quantile(0.25)
    Q3 = col.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return np.clip(col, lower, upper)

outlier_cols = ['Units Sold', 'Inventory Level', 'Price', 'Demand']

for col in outlier_cols:
    df[col] = cap_outliers(df[col])

In [13]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df['Day'] = df['Date'].dt.day
df['Quarter'] = df['Date'].dt.quarter

In [14]:
df['Net_Price'] = df['Price'] * (1 - df['Discount'] / 100)
df['Discount_Flag'] = np.where(df['Discount'] > 0, 1, 0)

In [15]:
df['Stock_Gap'] = df['Inventory Level'] - df['Units Sold']
df['Stockout_Risk'] = np.where(df['Stock_Gap'] < 0, 1, 0)

In [16]:
df['Demand_to_Inventory_Ratio'] = df['Demand'] / (df['Inventory Level'] + 1)

In [17]:
assert df.isnull().sum().sum() == 0, "Missing values found!"
assert (df['Units Sold'] >= 0).all(), "Negative sales detected!"
assert (df['Demand'] >= 0).all(), "Negative demand detected!"

print("✅ Data Quality Checks Passed")


✅ Data Quality Checks Passed


In [19]:
df.to_csv("../data/sales_data_cleaned.csv", index=False)